<a href="https://colab.research.google.com/github/one-2730/ESSA-25-1/blob/Assignment/ESAA_OB_0324_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#파머완 8장 텍스트 분석


NLP: 머신이 인간의 언어를 이해하고 해석하는 데 중점




텍스트 마이닝: 비정형 텍스트에서 의미있는 정보를 추출하는 데 중점
- 텍스트 분류
- 감성 분석
- 텍스트 요약
- 텍스트 군집화

##1. 텍스트 분석 이해

**텍스트 분석 수행 프로세스**
1. 텍스트 사전 준비작업(텍스트 전처리) : 클렌징, 대소문자 변경, 특수문자 삭제, 단어 등의 토큰화 작업, 의미없는 단어(stop word) 제거 작업, 어근 추출(Stemming/Lemmatization) 등의 텍스트 정규화 작업
2. 피처 벡터화/추출: BOW(Count 기반, TF-IDF 기반), Word2Vec
3. ML 모델 수립 및 학습/예측/평가



**NLP, 텍스트 분석 패키지**
- NLTK: 수행속도가 느려서 실제 대량 데이터 기반에서는 활용 X
- Genism: 토픽 모델링 분야에서 좋음
- SpaCy: 성능이 좋음

##2. 텍스트 전처리: 텍스트 정규화

###문장 토큰화

In [1]:
from nltk import sent_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

text_sample = 'The Matrix is everywhere its all around us, here even in this room. You can see it out your window or on your television. You feel it when you go to work, or go to church or pay you taxes.'
sentences = sent_tokenize(text = text_sample)
print(type(sentences), len(sentences))
print(sentences)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


<class 'list'> 3
['The Matrix is everywhere its all around us, here even in this room.', 'You can see it out your window or on your television.', 'You feel it when you go to work, or go to church or pay you taxes.']


###단어 토큰화

In [2]:
from nltk import word_tokenize

sentence = 'The Matrix is everywhere its all around us, here even in this room.'
words = word_tokenize(sentence)
print(type(words), len(words))
print(words)

<class 'list'> 15
['The', 'Matrix', 'is', 'everywhere', 'its', 'all', 'around', 'us', ',', 'here', 'even', 'in', 'this', 'room', '.']


In [3]:
def tokenize_text(text):
  sentences = sent_tokenize(text)
  word_tokenes = [word_tokenize(sentence) for sentence in sentences]
  return word_tokenes

word_tokens = tokenize_text(text_sample)
print(type(word_tokens), len(word_tokens))
print(word_tokens)

<class 'list'> 3
[['The', 'Matrix', 'is', 'everywhere', 'its', 'all', 'around', 'us', ',', 'here', 'even', 'in', 'this', 'room', '.'], ['You', 'can', 'see', 'it', 'out', 'your', 'window', 'or', 'on', 'your', 'television', '.'], ['You', 'feel', 'it', 'when', 'you', 'go', 'to', 'work', ',', 'or', 'go', 'to', 'church', 'or', 'pay', 'you', 'taxes', '.']]


###스톱 워드 제거

In [4]:
import nltk
nltk.download('stopwords')

print('영어 stop words 개수: ', len(nltk.corpus.stopwords.words('english')))
print(nltk.corpus.stopwords.words('english')[:20])

영어 stop words 개수:  198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
stopwords = nltk.corpus.stopwords.words('english')
all_tokens = []

for sentence in word_tokens:
  filtered_words = []

  for word in sentence:
    word = word.lower()
    if word not in stopwords:
      filtered_words.append(word)
  all_tokens.append(filtered_words)

print(all_tokens)

[['matrix', 'everywhere', 'around', 'us', ',', 'even', 'room', '.'], ['see', 'window', 'television', '.'], ['feel', 'go', 'work', ',', 'go', 'church', 'pay', 'taxes', '.']]


###Stemming, Lemmatization

In [6]:
from nltk.stem import LancasterStemmer
stemmer = LancasterStemmer()

print(stemmer.stem('working'), stemmer.stem('works'), stemmer.stem('worked'))
print(stemmer.stem('amusing'), stemmer.stem('amuses'), stemmer.stem('amused'))
print(stemmer.stem('happier'), stemmer.stem('happiest'))
print(stemmer.stem('fancier'), stemmer.stem('fanciest'))

work work work
amus amus amus
happy happiest
fant fanciest


In [7]:
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')

lemma = WordNetLemmatizer()
print(lemma.lemmatize('amusing', 'v'), lemma.lemmatize('amuses', 'v'), lemma.lemmatize('amused', 'v'))
print(lemma.lemmatize('happier', 'a'), lemma.lemmatize('happiest', 'a'))
print(lemma.lemmatize('fancier', 'a'), lemma.lemmatize('fanciest', 'a'))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


amuse amuse amuse
happy happy
fancy fancy


##3. Bag of Words

문장에서 단어가 등장하는 순서 등은 고려하지 않고, 사용된 단어의 빈도 등만을 유의미하다고 보는 가정


---


**사이킷런의 Count 및 TF-IDF 벡터화 구현: CountVectorizer, TfidfVectorizer**


: fit()과 transform() 을 통해 피처 벡터화된 객체를 반환

|파라미터 명|파라미터 설명|
-------|----|
max_df| 전체 문서에 걸쳐 n개 이하로 나타나는 단어만 피처로 추출 or 상위 a%만 추출 <br> : 너무 많이 등장하는 단어는 문법적 특성만 가질 뿐, 의미를 담지 않을 가능성이 높음|
min_df|너무 적게 나타나는 단어를 삭제|
max_features|추출하는 피처의 개수를 제한|
stop_words|'english'로 지정하면 영어의 스톱워드로 지정된 단어는 추출에서 제외|
n_gram_range|모델의 단어 순서를 어느정도 보강하기 위한 파라미터로 (범위 최솟값, 범위 최댓값) 지정|
analyzer|피처 추출을 수행한 단위. default = 'word'|
token_pattern|토큰화를 수행하는 정규표현식 패턴을 지정 default = '\b\w\w+\b'|
tokenizer| 토큰화를 별도의 커스텀 함수로 이용시 적용

###BOW 벡터화를 위한 희소 행렬

COO 형식

In [8]:
import numpy as np

dense = np.array([[3, 0, 1], [0, 2, 0]])

from scipy import sparse

data = np.array([3, 1, 2])

row_pos = np.array([0, 0, 1])
col_pos = np.array([0, 2, 1])

sparse_coo = sparse.coo_matrix((data, (row_pos, col_pos)))

sparse_coo.toarray()

array([[3, 0, 1],
       [0, 2, 0]])

CSR 형식

In [9]:
dense2 = np.array([[0, 0, 1, 0, 0, 5],
                   [1, 4, 0, 3, 2, 5],
                   [0, 6, 0, 3, 0, 0],
                   [2, 0, 0, 0, 0, 0],
                   [0, 0, 0, 7, 0, 8],
                   [1, 0, 0, 0, 0, 0]])

data2 = np.array([1, 5, 1, 4, 3, 2, 5, 6, 3, 2, 7, 8, 1])

row_pos = np.array([0, 0, 1, 1, 1, 1, 1, 2, 2, 3, 4, 4, 5])
col_pos = np.array([2, 5, 0, 1, 3, 4, 5, 1, 3, 0, 3, 5, 0])

sparse_coo = sparse.coo_matrix((data2, (row_pos, col_pos)))

row_pos_ind = np.array([0, 2, 7, 9, 10, 12, 13])

sparse_csr = sparse.csr_matrix((data2, col_pos, row_pos_ind))

print('COO 변환된 데이터가 제대로 되었는지 다시 Dense로 출력 확인')
print(sparse_coo.toarray())
print('CSR 변환된 데이터가 제대로 되었는지 다시 Dense로 출력 확인')
print(sparse_csr.toarray())

COO 변환된 데이터가 제대로 되었는지 다시 Dense로 출력 확인
[[0 0 1 0 0 5]
 [1 4 0 3 2 5]
 [0 6 0 3 0 0]
 [2 0 0 0 0 0]
 [0 0 0 7 0 8]
 [1 0 0 0 0 0]]
CSR 변환된 데이터가 제대로 되었는지 다시 Dense로 출력 확인
[[0 0 1 0 0 5]
 [1 4 0 3 2 5]
 [0 6 0 3 0 0]
 [2 0 0 0 0 0]
 [0 0 0 7 0 8]
 [1 0 0 0 0 0]]
